# Laboratorium 4: Transformer jako model

W poprzednich laboratoriach pracowaliśmy z TF-IDF i gotowymi embeddingami. W tym laboratorium przechodzimy do modelu transformerowego używanego bezpośrednio jako klasyfikator.

Śledzimy cały przepływ:

`tekst → tokenizator → input_ids + attention_mask → enkoder → reprezentacja → głowa klasyfikacyjna → logity → predykcja`

Główne pytanie laboratorium brzmi:

**Co daje dostrajanie transformera i kiedy wzrost jakości uzasadnia dodatkowy koszt względem prostszego baseline'u?**

Porównamy:

- `TF-IDF + Linear SVM` z laboratorium 2;
- transformer z zamrożonym enkoderem;
- ten sam transformer po pełnym dostrojeniu.

Laboratorium obejmuje 10 zadań obowiązkowych i jedno opcjonalne.

## Po tych zajęciach student potrafi

- wyjaśnić różnicę między słowem, tokenem i subwordem;
- odczytać `input_ids` i `attention_mask`;
- wyjaśnić rolę paddingu i obcinania sekwencji;
- odczytać podstawowe informacje o architekturze modelu;
- wykonać przejście w przód bez użycia `pipeline()`;
- zinterpretować `last_hidden_state`, logity i wynik softmax;
- przygotować dane do klasyfikacji tekstu transformerem;
- porównać zamrożony enkoder z pełnym dostrajaniem;
- porównać transformera z klasycznym baseline'em;
- porównać model wielojęzyczny z polskim pod względem tokenizacji, jakości i kosztu (opcjonalnie);
- przeprowadzić analizę błędów i uzasadnić decyzję projektową.

## Dane i protokół

Pracujemy na tym samym zbiorze PolEmo 2.0 co w laboratoriach 1-3.

Obowiązuje ten sam protokół:

- `train` -- uczenie;
- `validation` -- porównywanie wariantów i wybór konfiguracji;
- `test` -- jedna końcowa ocena wybranego rozwiązania.

Wynik `TF-IDF + Linear SVM` z laboratorium 2 jest punktem odniesienia. Nie zmieniamy baseline'u po zobaczeniu wyników transformera.

Jeżeli ograniczenia sprzętowe wymagają zmniejszenia liczby przykładów treningowych, należy ustawić TRAIN_LIMIT (np. 2000, próbka stratyfikowana, ta sama we wszystkich eksperymentach transformerowych) i opisać to w karcie modelu.

## Ustawienia wstępne

Wymagane biblioteki:

```bash
pip install pandas numpy matplotlib scikit-learn pyarrow torch transformers datasets accelerate
```

Model używany w laboratorium:

```text
distilbert/distilbert-base-multilingual-cased
```

W Zadaniu 8 (opcjonalnie): `allegro/herbert-base-cased`, model wstępnie wytrenowany na korpusach polskich.


Kod startowy obejmuje importy, stałe i wczytanie danych. Implementacja właściwych eksperymentów należy do Państwa.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import re

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

RANDOM_STATE = 42
set_seed(RANDOM_STATE)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
# MODEL_PL = "allegro/herbert-base-cased"   # zadanie 8 (opcjonalnie)
MAX_LENGTH = 128        # zadanie 2
TRAIN_LIMIT = None      # None = cały train; na CPU proponowane 2000

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None)
    and torch.backends.mps.is_available()
    else "cpu"
)

print("Urządzenie:", DEVICE)

In [ ]:
# helpery z labu 2 (bootstrap_ci, macro_f1) + nowe: paired_bootstrap_diff, token_stats

from sklearn.metrics import f1_score

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def bootstrap_ci(y_true, y_pred, metric, n_boot=1000, alpha=0.05, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        stats.append(metric(y_true[i], y_pred[i]))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(metric(y_true, y_pred)), float(lo), float(hi)
    
def paired_bootstrap_diff(y_true, pred_a, pred_b, metric, n_boot=1000, alpha=0.05, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        diffs.append(metric(y_true[i], pred_b[i]) - metric(y_true[i], pred_a[i]))
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(metric(y_true, pred_b) - metric(y_true, pred_a)), float(lo), float(hi)

def token_stats(tokenizer, texts, max_length):
    texts = list(texts)
    ids = tokenizer(texts, add_special_tokens=True, truncation=False)["input_ids"]
    lens = np.array([len(x) for x in ids])
    n_words = np.array([len(re.findall(r"\w+", t)) for t in texts])
    return {
        "median": float(np.median(lens)),
        "p90": float(np.percentile(lens, 90)),
        "pct_truncated": float((lens > max_length).mean()),
        "tokens_per_word": float(lens.sum() / n_words.sum()),
    }

In [ ]:
DATA_PATH = ARTIFACTS / "polemo2_clean.parquet"
assert DATA_PATH.exists(), "Brak artifacts/polemo2_clean.parquet. Proszę wykonać Lab 1."

data = pd.read_parquet(DATA_PATH).dropna(subset=["text", "target"]).reset_index(drop=True)
assert set(data["target"]) == {"plus", "minus", "zero", "amb"}

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df = data[data["split"].isin(["validation", "valid"])].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

if TRAIN_LIMIT is not None:
    train_df, _ = train_test_split(
        train_df, train_size=TRAIN_LIMIT, stratify=train_df["target"], random_state=RANDOM_STATE
    )
    train_df = train_df.reset_index(drop=True)

CLASSES = sorted(train_df["target"].unique())
print("train:", len(train_df), "validation:", len(val_df), "test:", len(test_df))

# Zadanie 1: Audyt tokenizacji

Proszę wczytać tokenizator `MODEL_NAME` i przygotować co najmniej 8 polskich przykładów obejmujących wybrane zróżnicowane zjawiska z podanej poniżej listy:

- częste słowo;
- rzadkie słowo;
- nazwę własną;
- nazwisko;
- wyraz z polskimi znakami;
- ten sam wyraz bez polskich znaków;
- literówkę;
- neologizm;
- emoji;
- URL;
- liczbę;
- termin specjalistyczny.

Dla każdego przykładu proszę zapisać:

| tekst | liczba słów | liczba tokenów | tokeny | obserwacja |
|---|---:|---:|---|---|

Proszę krótko odpowiedzieć:

1. Czy token odpowiada słowu?
2. Jak tokenizator zachowuje się wobec nietypowych form?
3. Dlaczego tokenizacja może wpływać na późniejszy wynik modelu?

In [ ]:
# Proszę wczytać tokenizator i przeprowadzić audyt tokenizacji.

# Zadanie 2: `input_ids`, `attention_mask`, padding i obcinanie

Proszę wybrać trzy teksty o różnej długości i przygotować je do wejścia modelu.

Dla każdego tekstu proszę wyświetlić:

- tokeny;
- `input_ids`;
- `attention_mask`;
- długość sekwencji.

Następnie proszę porównać `padding=True` oraz `padding="max_length", max_length=128` i policzyć, jaki procent pozycji w batchu stanowi padding.

Następnie proszę sprawdzić długości tokenowe wszystkich tekstów treningowych i podać:

- medianę;
- percentyl 90;
- percentyl 95;
- maksimum;
- procent tekstów obcinanych przy `MAX_LENGTH=128`.

Na tej podstawie proszę zdecydować, czy `128` jest rozsądną wartością dla dalszej części laboratorium (przy obcinaniu znika koniec tekstu, a w recenzjach często jest tam podsumowanie oceny). Decyzję i procent obciętych tekstów proszę zapisać w karcie modelu.

In [ ]:
# Proszę zaimplementować analizę wejścia modelu i długości tokenowych.

# Zadanie 3: Co znajduje się w modelu?

Proszę wczytać bazowy enkoder przez `AutoModel` i odczytać z konfiguracji:

- rozmiar słownika;
- wymiar stanu ukrytego;
- liczbę warstw;
- liczbę głów uwagi;
- maksymalną długość sekwencji.

Następnie proszę samodzielnie policzyć:

- wszystkie parametry modelu;
- parametry trenowalne;
- przybliżony rozmiar samych wag w FP32 i FP16.

Proszę krótko wyjaśnić, dlaczego rzeczywiste zużycie pamięci podczas treningu jest większe niż rozmiar samych wag.


In [ ]:
# Proszę wczytać AutoModel, odczytać konfigurację i policzyć parametry oraz przybliżony koszt pamięci wag.

# Zadanie 4: Ręczne przejście w przód

Proszę wybrać jeden tekst i wykonać `tekst → tokenizator → tensory → AutoModel → last_hidden_state` bez użycia `pipeline()`.

Należy:

1. podać kształty tensorów wejściowych;
2. podać kształt `last_hidden_state`;
3. wyjaśnić znaczenie każdego wymiaru;
4. przypisać tokeny do odpowiadających im reprezentacji;
5. utworzyć jeden wektor dokumentu przez mean pooling z uwzględnieniem `attention_mask`.

Następnie proszę wczytać `AutoModelForSequenceClassification`, wykonać pojedynczą predykcję przed treningiem i:

- odczytać logity;
- zastosować softmax;
- sprawdzić, czy wartości po softmax sumują się do 1.

Proszę pamiętać, że wynik softmax nie jest automatycznie skalibrowaną miarą pewności modelu.

Na końcu proszę wyjaśnić, dlaczego predykcja z losowo zainicjalizowaną głową klasyfikacyjną nie ma jeszcze sensownej jakości.


In [ ]:
# Proszę wykonać ręczne przejście w przód, mean pooling oraz analizę logitów i wyniku softmax.

Uwaga do zadań 3-4: Nazwy pól w konfiguracji DistilBERT różnią się od BERT-a (`dim`, `n_layers`, `n_heads`), ale działają też aliasy (`hidden_size`, `num_hidden_layers`, `num_attention_heads`).

Przy ładowaniu `AutoModelForSequenceClassification` pojawi się ostrzeżenie o nowo zainicjalizowanych wagach. Proszę wyjaśnić, które to wagi i dlaczego jest to oczekiwane.

Którą pozycję z `last_hidden_state` wykorzystuje głowa klasyfikacyjna tego modelu i czym różni się to od mean poolingu?

# Zadanie 5: Przygotowanie klasyfikatora

Proszę przygotować pełny pipeline treningowy dla klasyfikacji PolEmo 2.0.

Należy samodzielnie:

1. przygotować mapowania `label2id` i `id2label`;
2. utworzyć obiekty `Dataset` dla `train` i `validation`;
3. napisać funkcję tokenizującą batch;
4. użyć dynamicznego `DataCollatorWithPadding`;
5. zaimplementować `compute_metrics`;
6. przygotować `TrainingArguments`;
7. utworzyć `Trainer`.

Główną metryką jest `macro-F1`.

Konfiguracja domyślna (zmiana tylko z uzasadnieniem): 3 epoki, batch size 16, `weight_decay=0.01`, `warmup_ratio=0.1`, `MAX_LENGTH` z Zadania 2.

Proszę ustawić `report_to="none"` i `save_strategy="no"` (domyślnie `Trainer` zapisuje checkpointy po ok. 0,5 GB). Nazwy argumentów `TrainingArguments` i `Trainer` zmieniały się między wersjami `transformers` (m.in. `evaluation_strategy` → `eval_strategy`, `tokenizer` → `processing_class`), więc proszę zapisać `import transformers`.

Proszę zapisać użyte `MAX_LENGTH`, batch size, learning rate, liczbę epok, seed i urządzenie.

In [ ]:
# Proszę przygotować dane i pipeline treningowy.

# Zadanie 6: Zamrożony enkoder

Proszę załadować model klasyfikacyjny i zamrozić cały enkoder (`model.base_model`). Trenowana ma być wyłącznie głowa klasyfikacyjna (w DistilBERT: `pre_classifier` i `classifier`).

Learning rate dla tego wariantu proszę dobrać osobno (typowo ok. `1e-3`; wartość rzędu `2e-5` prawie nie uczy samej głowy w trzech epokach). Liczba epok i pozostałe warunki są takie same jak w zadaniu 5.

Proszę:

1. policzyć liczbę wszystkich parametrów;
2. policzyć liczbę parametrów trenowalnych;
3. podać procent parametrów trenowalnych;
4. wytrenować model;
5. zmierzyć czas treningu;
6. obliczyć accuracy i macro-F1 na zbiorze walidacyjnym.

Przed treningiem proszę zapisać hipotezę dotyczącą oczekiwanej jakości tego wariantu.

In [ ]:
# Proszę zaimplementować wariant z zamrożonym enkoderem.

# Zadanie 7: Pełne dostrajanie i porównanie na walidacji

Proszę załadować świeżą kopię tego samego checkpointu i przeprowadzić pełne dostrajanie modelu.

Warunki: ten sam podział danych, to samo `MAX_LENGTH`, ten sam sposób ewaluacji i ta sama liczba epok co w Zadaniu 5. Learning rate dobieramy osobno do wariantu (pełne dostrajanie: typowo `2e-5`–`5e-5`).

Dla wariantu dostrojonego proszę zmierzyć: liczbę parametrów trenowalnych, czas treningu, accuracy, macro-F1 i średni czas inferencji na dokument (ta sama maszyna, podany batch size).

| model | trainable params | train_s | ms/doc | accuracy | macro-F1 | macro-F1 CI95 |
|---|---:|---:|---:|---:|---:|---:|
| TF-IDF + Linear SVM | -- | | | | | |
| frozen transformer | | | | | | |
| fine-tuned transformer | | | | | | |

Przedział ufności liczymy funkcją `bootstrap_ci` z Lab 2. Predykcje walidacyjne obu transformerów proszę zapisać (będą potrzebne w Zadaniach 8 i 10).

Opcjonalnie: proszę powtórzyć dostrajanie dla drugiego seeda. Dostrajanie ma wyraźną wariancję między seedami, więc różnic rzędu kilku dziesiątych punktu macro-F1 nie należy traktować jako przewagi.

In [ ]:
# Proszę przeprowadzić pełne dostrajanie, zapisać predykcje walidacyjne i uzupełnić tabelę.

# (opcjonalnie) Zadanie 8: Model polski czy wielojęzyczny?

Dotychczas używaliśmy `distilbert-base-multilingual-cased` (6 warstw, słownik wielojęzyczny). Sprawdzimy, czy model wstępnie wytrenowany na polskich korpusach, `allegro/herbert-base-cased` (BERT, 12 warstw, słownik 50 tys. tokenów), daje przewagę i za jaką cenę.

## Część A: tokenizacja (bez treningu)

Proszę wczytać oba tokenizatory (HerBERT: `AutoTokenizer.from_pretrained(MODEL_PL)`) i dla 500 losowych dokumentów z `train_df` policzyć funkcją `token_stats`: medianę i p90 liczby tokenów, procent tekstów obciętych przy `MAX_LENGTH` oraz liczbę tokenów na słowo.

Następnie proszę porównać podział przykładów z Zadania 1 (także form z odciętą końcówką osobową, np. `Był em`, `miała m`) i zapisać, które słowa dzielą się inaczej.

## Część B: dostrajanie

Proszę dostroić HerBERT-a w tych samych warunkach co dostrojony DistilBERT z Zadania 7: te same `MAX_LENGTH`, liczba epok, batch size, seed i próbka danych, learning rate z tego samego przedziału. Przy ładowaniu `AutoModelForSequenceClassification` pojawi się ostrzeżenie o nowo zainicjalizowanych wagach. Proszę zapisać, które warstwy są nowe.

## Część C: porównanie

Proszę rozszerzyć tabelę z Zadania 7 o wiersz `fine-tuned HerBERT` (trainable params, train_s, ms/doc, accuracy, macro-F1, CI95) oraz policzyć różnicę macro-F1 (HerBERT − DistilBERT) z przedziałem ufności funkcją `paired_bootstrap_diff` na tych samych przykładach walidacyjnych.

Pytania:

1. Który model ma więcej parametrów, a który jest wolniejszy? Dlaczego liczba parametrów nie wystarcza do oszacowania kosztu obliczeń?
2. Czy różnica jakości jest istotna w świetle przedziału ufności?
3. Czy tokenizatory inaczej dzielą polskie formy (Część A)? Czy na podstawie tego eksperymentu można stwierdzić, że to wyjaśnia różnicę jakości?

In [ ]:
# Część A: porównanie tokenizatorów (token_stats)
# Część B: dostrojenie HerBERT-a (zapisać predykcje walidacyjne)
# Część C: tabela i paired_bootstrap_diff

# Zadanie 9*: Wybór finalisty i ostateczny test

Na podstawie wyłącznie wyników walidacyjnych (Zadania 6-8) proszę wybrać jeden wariant do końcowej oceny. Jeśli różnica między kandydatami mieści się w przedziale ufności, proszę to zaznaczyć i uzasadnić wybór kosztem.

Następnie:

1. proszę połączyć `train + validation`;
2. wytrenować wybrany wariant od początku (te same hiperparametry i liczba epok);
3. wykonać jedną ocenę na zbiorze testowym.

Proszę zapisać accuracy, macro-F1, F1 dla każdej klasy, macierz pomyłek i czas inferencji, a wynik porównać z wynikiem testowym `TF-IDF + Linear SVM` z Lab 2 (ten sam zbiór testowy).

Po obejrzeniu wyniku testowego konfiguracja nie może być już zmieniana.

Uwaga: model finalny widział `validation`, więc analiza błędów (zadanie 10) dotyczy modelu wytrenowanego wyłącznie na `train`.

In [ ]:
# Proszę wybrać finalistę na validation i wykonać jednorazowy finalny test.

# Zadanie 10*: Analiza błędnych predykcji

Dla najlepszego transformera wytrenowanego wyłącznie na `train` (nie dla modelu finalnego z zadania 9) proszę wybrać dokładnie 12 błędnych predykcji ze zbioru walidacyjnego (wylosowanych z `random_state=RANDOM_STATE`).

Każdy przypadek proszę przypisać do krótkiej kategorii, np. `AMBIGUOUS_GOLD`, `MIXED_SENTIMENT`, `NEGATION`, `IRONY_SARCASM`, `LONG_CONTEXT`, `VERY_SHORT`, `LEXICAL_TRAP`, `TRUNCATION`, `NON_REVIEW`, `PRETOKENIZATION`, `OTHER`.

Tabela powinna zawierać:

| nr | tekst | gold | predykcja | typ błędu | komentarz |
|---:|---|---|---|---|---|

Proszę dodatkowo zaznaczyć, jeśli w labie stworzyli Państwo takie check, czy TF-IDF + Linear SVM (predykcje z labu 2) popełnił ten sam błąd.

Nie należy jeszcze proponować rozwiązania każdego pojedynczego błędu. Najpierw proszę wskazać powtarzające się wzorce.

In [ ]:
# Proszę przygotować analizę dokładnie 12 błędnych predykcji.

# Zadanie 11*: Interpretacja, decyzja projektowa i wnioski

To zadanie nie wymaga nowej implementacji.

Na podstawie wykonanych eksperymentów proszę odpowiedzieć:

1. Co zmieniło pełne dostrajanie względem zamrożonego enkodera?
2. Czy wzrost jakości był proporcjonalny do wzrostu kosztu treningu?
3. Jakie typy błędów dominują w analizie 12 przypadków i czy któryś z nich prawdopodobnie nie ustąpi po zwiększeniu modelu?
4. Czy wynik tego laboratorium pozwala stwierdzić, że transformer jest ogólnie lepszy od `TF-IDF + Linear SVM`? Dlaczego?
5. (jeśli wykonano Zadanie 8) Czy model polski ma przewagę nad wielojęzycznym i czy ta przewaga uzasadnia koszt?

Następnie proszę wybrać rozwiązanie dla dwóch scenariuszy: **A** (najważniejsza jest jakość), **B** (najważniejsze są niski koszt, szybkie uczenie i szybka inferencja). Dla każdego proszę podać: wybrany model, wynik jakościowy, koszt do zaakceptowania i najważniejsze ograniczenie.

Na końcu jednym akapitem: **Kiedy dodatkowa złożoność transformera jest uzasadniona, a kiedy mocny model klasyczny pozostaje lepszą decyzją inżynierską?**ę

Proszę przygotować wnioski dotyczące zadania (5-8 krótkich wniosków opartych na wynikach, obejmujących tokenizację, MAX_LENGTH, frozen vs fine-tuning, SVM vs transformer, koszt i błędy).

# Karta modelu

Proszę utworzyć:

```text
artifacts/transformer_model_card.md
```

Karta powinna zawierać:

## Model
- checkpoint;
- typ architektury;
- (opcjonalnie) checkpoint modelu polskiego i liczba parametrów;
- liczba parametrów.

## Tokenizator
- checkpoint;
- `MAX_LENGTH`;
- procent tekstów obciętych przy `MAX_LENGTH`;
- (jeśli wykonano Zadanie 8) porównanie z tokenizatorem HerBERT-a (tokeny na słowo);
- sposób paddingu i obcinania.

## Zadanie
- przewidywane klasy;
- mapowanie `label2id` i `id2label`.

## Dane
- liczebność `train`, `validation`, `test`;
- ewentualne ograniczenie liczby przykładów z powodów sprzętowych.

## Trening
- batch size;
- learning rate;
- liczba epok;
- wariant: frozen / fine-tuned;
- liczba parametrów trenowalnych.

## Wyniki
- `TF-IDF + Linear SVM`;
- frozen transformer;
- fine-tuned transformer;
- fine-tuned HerBERT (jeśli wykonano);
- finalny wynik testowy.

## Koszt
- urządzenie;
- czas treningu;
- ms/dokument.

## Błędy
- dominujące typy błędów z analizy 12 przypadków.

## Zastosowanie
- kiedy użycie transformera jest uzasadnione;
- kiedy prostszy baseline może być lepszym wyborem.

## Ograniczenia
- ograniczenia danych;
- ograniczenia eksperymentu;
- czego wyniki tego laboratorium nie pozwalają stwierdzić.

# Warunki zaliczenia laboratorium

Laboratorium jest ukończone, jeśli zawiera:

- [ ] audyt tokenizacji dla co najmniej 8 przykładów;
- [ ] analizę `input_ids`, `attention_mask` i porównanie sposobów paddingu;
- [ ] analizę długości tokenowych i decyzję dotyczącą `MAX_LENGTH`;
- [ ] odczyt parametrów architektury, liczbę parametrów i koszt pamięci wag;
- [ ] ręczne przejście w przód bez `pipeline()` oraz mean pooling;
- [ ] analizę logitów i wyniku softmax;
- [ ] samodzielnie przygotowany pipeline treningowy;
- [ ] trening przy zamrożonym enkoderze (z osobno dobranym learning rate);
- [ ] pełne dostrajanie;
- [ ] tabelę porównawczą na validation z przedziałami ufności;
- [ ] porównanie z HerBERT-em, w tym paired bootstrap *(opcjonalnie)*;
- [ ] wybór konfiguracji na validation i jednorazową ocenę na test (z porównaniem do baseline'u z Lab 2);
- [ ] tabelę dokładnie 12 błędnych predykcji (model trenowany tylko na `train`);
- [ ] odpowiedzi do zadania interpretacyjnego (Zadanie 11);
- [ ] plik `artifacts/transformer_model_card.md`.